# Week 10: Four-Format Data Recovery and Validation

DSA4153 group project. This notebook runs the program in `src/` (it does not re-implement it),
then displays what the run produced: recovery counts, validation checks, the analysis tables,
the NumPy population index, the export round trips and the failure-handling tests.

Run it from the repository root after `pip install pandas numpy openpyxl pyarrow`.

## 1. Run the pipeline

In [1]:
import contextlib, io, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

from src.config import load_config
from src.pipeline import run_pipeline

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

cfg = load_config("config.json")
with contextlib.redirect_stdout(io.StringIO()):
    exit_code = run_pipeline(cfg)
out = Path(cfg["output_dir"])
print("exit code:", exit_code)
print("reporting_year:", cfg["reporting_year"])
print(sorted(p.name for p in out.iterdir()))

exit code: 0
reporting_year: 2007
['asia_selected_year.csv', 'audit_log.csv', 'canonical.csv', 'canonical.json', 'canonical.parquet', 'canonical.xlsx', 'country_history.csv', 'duplicate_log.csv', 'population_index.csv', 'quarantine.csv', 'validation.csv']


## 2. Input recovery and reconciliation

Each of the four inputs holds 1,712 records. They are read as text, validated, rejected records go to
`quarantine.csv`, exact duplicates are removed and logged in `duplicate_log.csv`, and the four tables
must agree before anything is published.

In [2]:
quarantine = pd.read_csv(out / "quarantine.csv", keep_default_na=False)
duplicates = pd.read_csv(out / "duplicate_log.csv", keep_default_na=False)
per_file = pd.DataFrame({
    "quarantined": quarantine.groupby("source_file").size(),
    "duplicates": duplicates.groupby("source_file").size(),
}).fillna(0).astype(int)
per_file["clean_rows"] = 1712 - per_file["quarantined"] - per_file["duplicates"]
per_file.loc["TOTAL"] = per_file.sum()
per_file

,quarantined,duplicates,clean_rows
source_file,,,
country_records.csv,6,2,1704
country_records.json,6,2,1704
country_records.parquet,6,2,1704
country_records.xlsx,6,2,1704
TOTAL,24,8,6816


In [3]:
quarantine["reason"].str.replace(r"'[^']*'", "'...'", regex=True).value_counts().to_frame("records")

,records
reason,
pop '...' must be a finite positive whole number,8
Year '...' is not a whole number in the allowed years,4
lifeExp '...' must be finite and between 0 and 120,4
gdpPercap '...' cannot be parsed as a number,4
Country cannot be empty,4


**Interpretation.** Every file lost the same 6 records to quarantine and the same 2 exact duplicates, so each
leaves 1,704 clean rows. Because four independently written readers (CSV, Excel, JSON, Parquet) end with the
same table, the recovery is consistent across formats. The 24 quarantined records are the deliberately planted
defects: 8 bad populations, and 4 each of blank country, invalid year, out-of-range life expectancy and
unparseable GDP. Nothing was silently repaired.

## 3. Validation checks (`validation.csv`)

In [4]:
validation = pd.read_csv(out / "validation.csv", keep_default_na=False)
print(f"{(validation['pass'].astype(str) == 'True').sum()} of {len(validation)} checks passed")
validation

44 of 44 checks passed


,check,expected,actual,tolerance,pass
0,input files match manifest SHA-256 hashes,0,0,0.000000e+00,True
1,csv: rows read = valid + quarantined,1712,1712,0.000000e+00,True
2,csv: no conflicting country/year keys,True,True,0.000000e+00,True
3,xlsx: rows read = valid + quarantined,1712,1712,0.000000e+00,True
4,xlsx: no conflicting country/year keys,True,True,0.000000e+00,True
5,json: rows read = valid + quarantined,1712,1712,0.000000e+00,True
6,json: no conflicting country/year keys,True,True,0.000000e+00,True
7,parquet: rows read = valid + quarantined,1712,1712,0.000000e+00,True
8,parquet: no conflicting country/year keys,True,True,0.000000e+00,True
9,total quarantined records,24,24,0.000000e+00,True


In [5]:
agreement = validation[validation["check"].str.contains("agree|matches|equals|conflicting", case=False)]
print("agreement and consistency checks:", len(agreement), "| all passed:", bool((agreement["pass"].astype(str) == "True").all()))
agreement[["check", "expected", "actual", "pass"]]

agreement and consistency checks: 15 | all passed: True


,check,expected,actual,pass
2,csv: no conflicting country/year keys,True,True,True
4,xlsx: no conflicting country/year keys,True,True,True
6,json: no conflicting country/year keys,True,True,True
8,parquet: no conflicting country/year keys,True,True,True
11,recovered tables agree,0,0,True
22,1982 baseline column equals 100,True,True,True
23,broadcast index equals independent loop,True,True,True
30,csv: row count matches canonical,1704,1704,True
31,csv: reloaded data matches canonical,0,0,True
34,xlsx: row count matches canonical,1704,1704,True


**Interpretation.** The four recovered tables agree with each other (key by key, floats within 1e-8, no relative
tolerance), no country/year key has conflicting values, and each reloaded export matches the pre-export table.
A conflicting key (`data/conflicting_key.csv`) stops the run instead of being resolved by guessing; the failure
tests below show this.

## 4. Audit log (`audit_log.csv`)

In [6]:
pd.read_csv(out / "audit_log.csv", keep_default_na=False)

,step,source_file,operation,rule,rows_before,rows_after
0,1,manifest.json,integrity check,SHA-256 and size of each input file,4.0,4
1,2,country_records.csv,read,csv reader,,1712
2,3,country_records.csv,validate,quarantine invalid records,1712.0,1706
3,4,country_records.csv,deduplicate,drop exact 6-field duplicates,1706.0,1704
4,5,country_records.xlsx,read,xlsx reader,,1712
5,6,country_records.xlsx,validate,quarantine invalid records,1712.0,1706
6,7,country_records.xlsx,deduplicate,drop exact 6-field duplicates,1706.0,1704
7,8,country_records.json,read,json reader,,1712
8,9,country_records.json,validate,quarantine invalid records,1712.0,1706
9,10,country_records.json,deduplicate,drop exact 6-field duplicates,1706.0,1704


## 5. pandas analysis

Asia in the configured year, selected with `.loc`, population in millions and a descending
life-expectancy rank (`method="min"`), sorted by rank then country.

In [7]:
asia = pd.read_csv(out / "asia_selected_year.csv")
print(len(asia), "rows; population total:", f"{asia['pop'].sum():,}")
asia.head(10)

33 rows; population total: 3,811,953,827


,country,continent,year,lifeExp,pop,gdpPercap,pop_millions,lifeExp_rank
0,Japan,Asia,2007,82.603,127467972,31656.068060,127.467972,1
1,"Hong Kong, China",Asia,2007,82.208,6980412,39724.978670,6.980412,2
2,Israel,Asia,2007,80.745,6426679,25523.277100,6.426679,3
3,Singapore,Asia,2007,79.972,4553009,47143.179640,4.553009,4
4,"Korea, Rep.",Asia,2007,78.623,49044790,23348.139730,49.044790,5
5,Taiwan,Asia,2007,78.400,23174294,28718.276840,23.174294,6
6,Kuwait,Asia,2007,77.588,2505559,47306.989780,2.505559,7
7,Oman,Asia,2007,75.640,3204897,22316.192870,3.204897,8
8,Bahrain,Asia,2007,75.635,708573,29796.048340,0.708573,9
9,Vietnam,Asia,2007,74.249,85262356,2441.576404,85.262356,10


**Interpretation.** Japan has the highest 2007 life expectancy in Asia (82.6 years), followed by Hong Kong and
Israel. The 33 Asian countries total 3,811,953,827 people, which matches the expected value; the same total is
computed independently with a NumPy Boolean selection and compared in `validation.csv`. Ties share a rank
(`method="min"`). Changing `reporting_year` in `config.json` (for example to 1997) changes this table.

In [8]:
history = pd.read_csv(out / "country_history.csv")
print(len(history), "Philippine observations")
history[["country", "year", "lifeExp", "pop", "pop_millions", "lifeExp_change"]]

12 Philippine observations


,country,year,lifeExp,pop,pop_millions,lifeExp_change
0,Philippines,1952,47.752,22438691,22.438691,0.000
1,Philippines,1957,51.334,26072194,26.072194,3.582
2,Philippines,1962,54.757,30325264,30.325264,7.005
3,Philippines,1967,56.393,35356600,35.356600,8.641
4,Philippines,1972,58.065,40850141,40.850141,10.313
5,Philippines,1977,60.060,46850962,46.850962,12.308
6,Philippines,1982,62.082,53456774,53.456774,14.330
7,Philippines,1987,64.151,60017788,60.017788,16.399
8,Philippines,1992,66.458,67185766,67.185766,18.706
9,Philippines,1997,68.564,75012988,75.012988,20.812


**Interpretation.** The Philippines gained 23.9 years of life expectancy between 1952 and 2007 (47.8 to 71.7), and
its population grew about fourfold, from 22.4 million to 91.1 million. All 12 observations are present and sorted by
year, and the 2007 population matches the expected 91,077,287.

## 6. NumPy population index (1982 = 100)

A 3 x 6 matrix (Philippines, Indonesia, Thailand by 1982, 1987, ..., 2007). Each row's 1982 baseline
(shape 3 x 1) is broadcast against the matrix (3 x 6) to give the index. The result is checked against an
independently written loop.

In [9]:
from src.analysis import population_matrix, index_by_broadcast, index_by_loop
canonical = pd.read_csv(out / "canonical.csv", keep_default_na=False)
matrix = population_matrix(canonical)
baseline, index = index_by_broadcast(matrix)
print("matrix", matrix.shape, "| baseline", baseline.shape, "| broadcast result", index.shape)
print("matches independent loop:", np.allclose(index, index_by_loop(matrix), atol=1e-8, rtol=0.0))
pd.read_csv(out / "population_index.csv")

matrix (3, 6) | baseline (3, 1) | broadcast result (3, 6)
matches independent loop: True


,country,1982,1987,1992,1997,2002,2007
0,Philippines,100.0,112.273494,125.682418,140.324570,155.256447,170.375577
1,Indonesia,100.0,110.390432,120.524576,129.955720,137.639149,145.782331
2,Thailand,100.0,108.362522,116.056504,123.326192,128.630762,133.262203


Boolean selection and aggregation on the same matrix: which countries ended above their 1982 level,
and their mean 2007 index.

**Interpretation.** Indexed to 1982 = 100, the Philippines grew fastest (170.4 in 2007), then Indonesia (145.8) and
Thailand (133.3). The broadcast calculation (3 x 6 divided by 3 x 1) matches an independently written loop, and
the Boolean selection and aggregation both give the expected result.

In [10]:
countries = ["Philippines", "Indonesia", "Thailand"]
grew = index[:, -1] > 100.0
print("countries above 100 in 2007:", [c for c, g in zip(countries, grew) if g])
print("mean 2007 index of those:", round(float(index[grew, -1].mean()), 2))

countries above 100 in 2007: ['Philippines', 'Indonesia', 'Thailand']
mean 2007 index of those: 149.81


## 7. Canonical exports and reload

The canonical table is exported as CSV, Excel (sheet `observations`), JSON (records) and Parquet.
Each export is reloaded, the declared schema is reapplied, and it is compared with the pre-export table.

In [11]:
from src.exports import read_exports, apply_schema, types_ok, COLUMNS
from src.compare import compare_tables

original = canonical[COLUMNS].copy()
original, _ = apply_schema(original)
rows = []
for fmt, reloaded in read_exports(out).items():
    normalized, conversions = apply_schema(reloaded[COLUMNS])
    rows.append({"format": fmt, "rows": len(normalized), "types_ok": types_ok(normalized),
                 "differences": len(compare_tables({"canonical": original, fmt: normalized})),
                 "conversions": "; ".join(conversions) or "none"})
pd.DataFrame(rows)

,format,rows,types_ok,differences,conversions
0,csv,1704,True,0,none
1,xlsx,1704,True,0,none
2,json,1704,True,0,none
3,parquet,1704,True,0,none


**Interpretation.** All four exports reload to exactly the 1,704 canonical rows with the declared types, so no type
conversion was needed on reload and no information was lost in any format.

## 8. Failure handling

The 12 required categories are exercised by `tests/test_step5.py`, each on a temporary copy of the data.

In [12]:
result = subprocess.run([sys.executable, "-m", "unittest", "-v", "tests.test_step5"],
                        capture_output=True, text=True)
lines = [l for l in result.stderr.splitlines() if " ... " in l or l.startswith(("Ran", "OK", "FAILED"))]
print("\n".join(l.split(" (")[0] + (" ... " + l.rsplit(" ... ", 1)[1] if " ... " in l else "") if " ... " in l else l for l in lines))

test_01_normal_four_format_recovery ... ok
test_02_missing_file ... ok
test_03_unsupported_format ... ok
test_04_missing_column ... ok
test_05_missing_key ... ok
test_06_unparseable_number ... ok
test_07_fractional_year_and_population ... ok
test_08_non_finite_and_out_of_range ... ok
test_09_exact_duplicates ... ok
test_10_conflicting_keys ... ok
test_11_reordered_records ... ok
test_12_corrupted_export_fails_comparison ... /Users/donderekmagpantay/projects/week10-data-recovery/src/exports.py:58: Pandas4Warning: The 'convert_dates' keyword in read_json is deprecated and will be removed in a future version. Pass dtype=False to disable type conversion, or parse date columns with pd.to_datetime after reading.
test_13_bad_row_quarantined_in_normal_run ... ok
test_16_changed_input_byte_stops_run ... ok
test_17_normal_run_records_integrity_check ... ok
test_14_unchanged_inputs_pass ... ok
test_15_one_changed_byte_is_reported ... ok
Ran 17 tests in 6.081s


**Interpretation.** The 12 required categories all behave as specified: file and schema errors and conflicting keys
stop the run with a nonzero exit and publish nothing, invalid individual records are quarantined during a normal
run, reordered records give identical output, and a deliberately altered export fails the comparison. The extra
tests cover the input integrity check on the supplied files.

## 9. Git contribution evidence

Per-member responsibilities, commits and reviews are recorded in `README.md`, `contributions.csv` and `reviews.md`.